In [ ]:
from pathlib import Path


def find_project_root(start: Path = Path.cwd()) -> Path:
    """Find project root by walking up to README.md and data/."""
    for path in [start, *start.parents]:
        if (path / 'README.md').exists() and (path / 'data').exists():
            return path
    raise FileNotFoundError(
        'Project root not found. Run this notebook from inside the project folder.'
    )


PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
FIGURES_DIR = PROJECT_ROOT / 'figures'
MODELS_DIR = PROJECT_ROOT / 'models'
NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'

SLEEP_EDF_DIR = RAW_DIR / 'sleep_edf' / 'sleep-cassette'

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('SLEEP_EDF_DIR =', SLEEP_EDF_DIR)
print('SLEEP_EDF_DIR exists =', SLEEP_EDF_DIR.exists())


In [ ]:
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

from scipy.signal import welch
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    classification_report
)

In [ ]:
epoch_df = pd.read_csv(PROCESSED_DIR / 'sleep_edf_epoch_index.csv')

def get_subject_prefix(filename):
    m = re.match(r'^(SC\d{4})', filename)
    return m.group(1) if m else None

psg_files = {}

for f in sorted(SLEEP_EDF_DIR.glob('*-PSG.edf')):
    prefix = get_subject_prefix(f.name)
    if prefix is not None:
        psg_files[prefix] = f

print(epoch_df.shape)
print(sorted(epoch_df['subject_id'].unique()))
print(sorted(psg_files.keys()))
epoch_df.head()

In [ ]:
print(epoch_df['stage_label'].value_counts())
print()
print(epoch_df['subject_id'].value_counts())

In [ ]:
def choose_channel(ch_names, preferred_patterns):
    for pattern in preferred_patterns:
        for ch in ch_names:
            if pattern.lower() in ch.lower():
                return ch
    return None


def bandpower(signal, sfreq, fmin, fmax):
    freqs, psd = welch(signal, fs=sfreq, nperseg=min(len(signal), int(sfreq * 4)))
    mask = (freqs >= fmin) & (freqs < fmax)
    if mask.sum() == 0:
        return np.nan
    return np.trapezoid(psd[mask], freqs[mask])


def spectral_entropy(signal, sfreq):
    freqs, psd = welch(signal, fs=sfreq, nperseg=min(len(signal), int(sfreq * 4)))
    psd = np.maximum(psd, 1e-12)
    p = psd / psd.sum()
    return -(p * np.log(p)).sum()


def extract_epoch_signal_features(signal, sfreq):
    signal = np.asarray(signal, dtype=float)

    delta = bandpower(signal, sfreq, 0.5, 4)
    theta = bandpower(signal, sfreq, 4, 8)
    alpha = bandpower(signal, sfreq, 8, 12)
    sigma = bandpower(signal, sfreq, 12, 15)
    beta = bandpower(signal, sfreq, 15, 30)

    total = np.nansum([delta, theta, alpha, sigma, beta])

    return {
        'mean': np.mean(signal),
        'std': np.std(signal),
        'ptp': np.ptp(signal),
        'rms': np.sqrt(np.mean(signal ** 2)),
        'delta': delta,
        'theta': theta,
        'alpha': alpha,
        'sigma': sigma,
        'beta': beta,
        'delta_rel': delta / total if total > 0 else np.nan,
        'theta_rel': theta / total if total > 0 else np.nan,
        'alpha_rel': alpha / total if total > 0 else np.nan,
        'sigma_rel': sigma / total if total > 0 else np.nan,
        'beta_rel': beta / total if total > 0 else np.nan,
        'spectral_entropy': spectral_entropy(signal, sfreq),
    }

In [ ]:
def extract_subject_epoch_features(subject_id, subject_epoch_df):
    if subject_id not in psg_files:
        raise FileNotFoundError(f'PSG file not found for {subject_id}')

    psg_path = psg_files[subject_id]

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        raw = mne.io.read_raw_edf(psg_path, preload=False, verbose=False)

    sfreq = raw.info['sfreq']
    eeg_ch = choose_channel(raw.ch_names, ['EEG Fpz-Cz', 'EEG Pz-Oz', 'EEG'])

    print(f'Processing {subject_id} | EEG channel: {eeg_ch}')

    if eeg_ch is None:
        raise ValueError(f'No EEG channel found for {subject_id}')

    eeg_data = raw.get_data(picks=[eeg_ch])[0]

    feature_rows = []

    for row in subject_epoch_df.itertuples(index=False):
        start_sample = int(row.epoch_start_sec * sfreq)
        stop_sample = int((row.epoch_start_sec + row.epoch_duration_sec) * sfreq)

        signal = eeg_data[start_sample:stop_sample]
        feats = extract_epoch_signal_features(signal, sfreq)

        feats['subject_id'] = row.subject_id
        feats['epoch_start_sec'] = row.epoch_start_sec
        feats['epoch_duration_sec'] = row.epoch_duration_sec
        feats['stage_label'] = row.stage_label

        feature_rows.append(feats)

    return pd.DataFrame(feature_rows)

In [ ]:
test_subject_id = epoch_df['subject_id'].iloc[0]
test_subject_epoch_df = epoch_df[epoch_df['subject_id'] == test_subject_id].copy()

print('test_subject_id =', test_subject_id)
print('n epochs =', len(test_subject_epoch_df))
print('starting feature extraction...')

test_features_df = extract_subject_epoch_features(test_subject_id, test_subject_epoch_df)

print('feature extraction done')
print(test_features_df.shape)
test_features_df.head()

In [ ]:
import time

subject_ids = sorted(epoch_df['subject_id'].unique())[:2]

print('Will process:', subject_ids, flush=True)

all_feature_dfs = []

for subject_id in subject_ids:
    t0 = time.time()
    print(f'\n=== START {subject_id} ===', flush=True)

    subject_epoch_df = (
        epoch_df[epoch_df['subject_id'] == subject_id]
        .sample(n=200, random_state=42)
        .sort_values('epoch_start_sec')
        .copy()
    )

    print(f'n epochs = {len(subject_epoch_df)}', flush=True)
    print(subject_epoch_df['stage_label'].value_counts(), flush=True)

    subject_features_df = extract_subject_epoch_features(subject_id, subject_epoch_df)

    print(f'{subject_id} done in {time.time() - t0:.1f} sec | shape = {subject_features_df.shape}', flush=True)
    all_feature_dfs.append(subject_features_df)

features_df = pd.concat(all_feature_dfs, ignore_index=True)

print('\nFINAL SHAPE:', features_df.shape, flush=True)
print('\nStage distribution in features_df:')
print(features_df['stage_label'].value_counts())
features_df.head()

In [ ]:
features_path = PROCESSED_DIR / 'sleep_edf_epoch_features.csv'
features_df.to_csv(features_path, index=False)

print(features_path.relative_to(PROJECT_ROOT) if features_path != PROJECT_ROOT else PROJECT_ROOT.name)
print(features_path.exists())

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, balanced_accuracy_score, cohen_kappa_score

feature_cols = [
    c for c in features_df.columns
    if c not in ['subject_id', 'epoch_start_sec', 'epoch_duration_sec', 'stage_label']
]

X = features_df[feature_cols].copy()
y = features_df['stage_label'].copy()
groups = features_df['subject_id'].copy()

print(X.shape)
print(y.shape)
print(sorted(groups.unique()))

In [ ]:
logo = LeaveOneGroupOut()

rf_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight='balanced_subsample'
    ))
])

pred = cross_val_predict(rf_model, X, y, cv=logo, groups=groups)

print('done')

In [ ]:
results_df = pd.DataFrame([{
    'model': 'RandomForest_mini',
    'macro_f1': f1_score(y, pred, average='macro'),
    'weighted_f1': f1_score(y, pred, average='weighted'),
    'balanced_accuracy': balanced_accuracy_score(y, pred),
    'cohen_kappa': cohen_kappa_score(y, pred)
}])

results_df

In [ ]:
results_path = PROCESSED_DIR / 'sleep_staging_baseline_results.csv'
results_df.to_csv(results_path, index=False)

print(results_path.relative_to(PROJECT_ROOT))
print(results_path.exists())

In [ ]:
stage_order = ["W", "N1", "N2", "N3", "REM"]

available_stage_order = [
    stage for stage in stage_order
    if stage in set(y) or stage in set(pred)
]

available_stage_order

In [ ]:
print(
    classification_report(
        y,
        pred,
        labels=available_stage_order,
        zero_division=0
    )
)

In [ ]:
predictions_df = features_df[['subject_id', 'epoch_start_sec', 'stage_label']].copy()
predictions_df['predicted_stage'] = pred

preds_path = PROCESSED_DIR / 'sleep_staging_cv_predictions.csv'
predictions_df.to_csv(preds_path, index=False)

print(preds_path)
print(preds_path.exists())
predictions_df.head()

In [ ]:
results_path = PROCESSED_DIR / 'sleep_staging_baseline_results_mini.csv'
results_df.to_csv(results_path, index=False)

predictions_df = features_df[['subject_id', 'epoch_start_sec', 'stage_label']].copy()
predictions_df['predicted_stage'] = pred

preds_path = PROCESSED_DIR / 'sleep_staging_cv_predictions_mini.csv'
predictions_df.to_csv(preds_path, index=False)

print(results_path.relative_to(PROJECT_ROOT), results_path.exists())
print(preds_path.relative_to(PROJECT_ROOT), preds_path.exists())

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt

stage_order = ['W', 'N1', 'N2', 'N3', 'REM']
cm = confusion_matrix(y, pred, labels=stage_order)

plt.figure(figsize=(7, 6))
plt.imshow(cm, aspect='auto')
plt.xticks(range(len(stage_order)), stage_order)
plt.yticks(range(len(stage_order)), stage_order)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Baseline Sleep Staging — RandomForest (mini)')
plt.colorbar()
plt.tight_layout()

fig_path = FIGURES_DIR / 'baseline_confusion_matrix_mini.png'
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()

stage_order = ["W", "N1", "N2", "N3", "REM"]

available_stage_order = [
    stage for stage in stage_order
    if stage in set(y) or stage in set(pred)
]

available_stage_order

print(fig_path.relative_to(PROJECT_ROOT), fig_path.exists())
print(
    classification_report(
        y,
        pred,
        labels=available_stage_order,
        zero_division=0
    )
)

In [ ]:
report_dict = classification_report(y, pred, labels=stage_order, zero_division=0, output_dict=True)
report_df = pd.DataFrame(report_dict).T
report_df

In [ ]:
report_path = PROCESSED_DIR / 'sleep_staging_classification_report_mini.csv'
report_df.to_csv(report_path, index=True)

print(report_path.relative_to(PROJECT_ROOT) if report_path != PROJECT_ROOT else PROJECT_ROOT.name)
print(report_path.exists())

## Baseline model interpretation

This mini-baseline sleep staging experiment shows that the pipeline is working and produces plausible stage-wise performance patterns.

The model performs best on Wake and N3, achieves moderate performance on N2, and struggles with REM and especially N1. This is expected for a small baseline model because N1 is both rare and difficult to separate from neighboring stages such as Wake and N2.

Overall, the results suggest that simple spectral EEG features already contain useful sleep-stage information, while performance on minority and transition-related stages remains limited.

## Conclusion

A baseline Random Forest model trained on epoch-level EEG spectral features was able to recover meaningful sleep-stage structure from Sleep-EDF mini-samples.

The strongest performance was observed for Wake and N3, while N1 remained the most difficult stage. This provides a reasonable baseline for the next stage of the project: deriving sleep fragmentation metrics from true and predicted stage sequences.

## Interpretation

This mini-baseline validates the core sleep staging pipeline on a small subset of Sleep-EDF data. The model shows plausible stage-wise behavior, with stronger performance on Wake and N3 and weaker performance on N1 and REM.

These results are best interpreted as a proof of pipeline functionality rather than a final benchmark.